# Nome do Aluno: Milton Salgado Leandro
# Instituição de Ensino: Universidade Federal do Rio de Janeiro

# 1.3 — Tratamento dos Dados (Silver)

**Objetivo:** padronizar os dados da Bronze em sete tabelas Delta, sem alterar a camada de origem. Aplicam-se tipagem segura, normalização, regras de domínio e deduplicação.

**Execução:** a cotação (`1.3.7`) antecede o financeiro (`1.3.2`) por dependência. A Silver utiliza `Overwrite`, permitindo reconstrução após correções na Bronze.

**Identidade do filme:** a unicidade será definida pelo `id` TMDB. IDs TMDB distintos que compartilham `tconst` IMDb são uma limitação dos dados de origem e não serão mesclados implicitamente.


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOGO = "workspace"

spark.sql(f"USE CATALOG {CATALOGO}")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")


# Salvar tabela Silver em formato Delta.
# Overwrite evita acumular duplicatas quando
# executamos novamente o notebook Silver.
def salvar_silver(df, nome):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"silver.{nome}")
    )


# Deduplicação:
# preservar a ingestão mais recente por filme.
def mais_recente(df):

    colunas = sorted([
        c for c in df.columns
        if c != "ingestion_datetime"
    ])

    janela = (
        Window.partitionBy("id")
        .orderBy(
            F.col("ingestion_datetime").desc(),
            F.xxhash64(
                *[F.col(c) for c in colunas]
            ).desc()
        )
    )

    return (
        df
        .filter(
            F.col("id").rlike(r"^[0-9]+$")
        )
        .withColumn(
            "_rn", F.row_number().over(janela)
        )
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

# 1.3.1 — Informações dos Filmes

**Origem:** `bronze.tb_movies_info`  
**Destino:** `silver.tb_info_filmes`

| Origem | Destino |
|---|---|
| `id` | `id_filme` |
| `title` / `original_title` | `titulo` / `titulo_original` |
| `release_date` / `runtime` | `data_lancamento` / `duracao_minutos` |
| `original_language` / `status` | `idioma_original` / `status_filme` |
| `overview` / `tagline` | `sinopse` / `frase_divulgacao` |

**Regras:** selecionar a ingestão mais recente por `id`, normalizar e traduzir status (não mapeados → `Não Informado`), interpretar formatos de data, converter duração para `INT` e derivar `ano_lancamento`. Datas ambíguas com barras priorizam `dd/MM/yyyy`.


In [0]:
# Origem Bronze e deduplicação.
df = mais_recente(
    spark.table("bronze.tb_movies_info")
)

# Normalização dos status.
status = F.lower(
    F.trim(
        F.regexp_replace(
            F.col("status"),
            r"[^A-Za-z]+", " "
        )
    )
)

traducao = {
    "released": "Lançado",
    "post production": "Pós-Produção",
    "in production": "Em Produção",
    "planned": "Planejado",
    "rumored": "Rumores",
    "canceled": "Cancelado"
}

mapa = F.create_map(*[
    item
    for chave, valor in traducao.items()
    for item in (F.lit(chave), F.lit(valor))
])

df = df.withColumn(
    "status_filme",
    F.coalesce(
        F.element_at(mapa, status),
        F.lit("Não Informado")
    )
)

# Conversão robusta das datas.
# Para barras, priorizamos dia/mês/ano.
# Para hífens, priorizamos ano-mês-dia
# ou mês-dia-ano, conforme o padrão.
formatos = [
    "yyyy-MM-dd",
    "MM-dd-yyyy",
    "dd/MM/yyyy",
    "MM/dd/yyyy",
    "dd-MM-yyyy",
    "yyyy/MM/dd"
]

datas = [
    F.expr(
        f"try_to_timestamp(release_date, '{fmt}')"
    ).cast("date")
    for fmt in formatos
]

df = df.withColumn(
    "data_lancamento",
    F.coalesce(*datas)
)

# Seleção, tipagem e renomeação.
resultado = df.select(
    F.col("id").cast("string").alias(
        "id_filme"
    ),
    F.col("title").alias("titulo"),
    F.col("original_title").alias(
        "titulo_original"
    ),
    "data_lancamento",
    F.expr(
        "try_cast(runtime AS INT)"
    ).alias("duracao_minutos"),
    F.col("original_language").alias(
        "idioma_original"
    ),
    "status_filme",
    F.col("overview").alias("sinopse"),
    F.col("tagline").alias(
        "frase_divulgacao"
    )
)

# Coluna derivada.
resultado = resultado.withColumn(
    "ano_lancamento",
    F.year("data_lancamento")
)

salvar_silver(resultado, "tb_info_filmes")

display(resultado.limit(20))

id_filme,titulo,titulo_original,data_lancamento,duracao_minutos,idioma_original,status_filme,sinopse,frase_divulgacao,ano_lancamento
1000004,Purple Beatz,Purple Beatz,2022-07-07,86,en,Lançado,"Sarah-Jane is a young aspiring jazz singer from Bournemouth who moves to London to embark on a music career. Not long in town, she falls for the handsome Airbeats, but also sinister music producer Russell-D, who represents a darker side to the music industry.",A drum n bass romance.,2022
1000005,Aisha Brown: The First Black Woman Ever,Aisha Brown: The First Black Woman Ever,2020-02-14,42,en,Lançado,null,null,2020
1000007,KYLE BROWNRIGG: INTRODUCING LYLE,Kyle Brownrigg: Introducing Lyle,2022-05-27,36,en,Lançado,"Kyle Brownrigg takes the stage in this hilarious half-hour stand-up special where he laments gender reveal parties, talks about his Irish boyfriend, and introduces the world to his drunk persona, Lyle.",null,2022
1000011,Worth Your Weight in Gold,O Teu Peso Em Ouro,2022-07-14,26,pt,Lançado,"Oscar, a renowned hypnotherapist, uses the last moments in his hotel room to say goodbye to the vertiginous Mercedes and rid old Norberto of his bitterness.",null,2022
1000014,On va manquer !,On va manquer !,2018-05-15,0,fr,Lançado,null,null,2018
1000030,58 Hours: The Baby Jessica Story,58 Hours: The Baby Jessica Story,2021-07-31,0,es,Lançado,null,null,2021
1000054,One Hundred Years and Hope,百年と希望,2022-06-18,107,ja,Lançado,"In a country ruled by the Liberal Democratic Party, running on austerity and neoliberal ambitions, for most of its postwar years, gender and economic inequalities have become increasingly acute in Japan. Takashi Nishihara, a filmmaker who has been following the youth protests in Japan notices that there is one party that seems to be raising issues of gender and economic in the political sphere, the Japanese Communist Party (JCP), a party about to enter its hundredth year and consistently burdened by its historical connotations. Though an outsider of the party, Nishihara gained unprecedented access to the JCP and driven by his interest in the younger party members who find hope in the JCP, the resulting documentary goes beyond party politics and observes the current grassroots leftist movements in Japan. It also becomes witness to the larger and deep-seated patriarchal system that continues to quell momentums of hope.",null,2022
1000058,Homecoming,Le retour,2023-07-12,110,fr,Lançado,"Kheìdidja, in her forties, works for a wealthy Parisian family who offers her the opportunity to take care of their children for a summer in Corsica. It's an opportunity for her to return with her daughters, Jessica and Farah, to the island they left fifteen years earlier in tragic circumstances.",null,2023
1000059,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,素敵な選TAXI SPECIAL〜湯けむり連続選択肢〜,2016-04-05,116,ja,Lançado,"Edawakare, the driver of the Time Taxi that allows passengers to return to their life's turning points, visits a hot spring this time around. An array of guests at the inn are fraught with troubles in their life and become passengers of the Time Taxi. And somehow all the clients are actually part of a bigger picture?!",null,2016
1000073,A Chance To Win,Pour l'honneur,2023-05-03,97,fr,Lançado,"Two villages in the south of France have always been bitter rivals, but when a group of asylum seekers arrive in the community, the life of both villages is shaken up and age-old disagreements escalate. Their antagonism reaches its peak with the annual rugby derby played between the two village teams, but this time, with the new outsiders joining as unexpected recruits, the result of the 100th match will be more unpredictable than ever.",null,2023


# 1.3.7 — Cotação do Dólar (pré-requisito do financeiro)

**Origem:** `bronze.tb_cotacao_dolar`  
**Destino:** `silver.tb_cotacao_dolar`

Converter data e cotação, manter a última cotação do dia e construir calendário contínuo. Preencher dias sem registro pelo último valor conhecido (**Forward Fill**). A janela global, aqui aplicada a poucos dias, pode emitir aviso de desempenho no Spark.


In [0]:
df = (
    spark.table("bronze.tb_cotacao_dolar")
    .withColumn(
        "instante",
        F.expr(
            "try_cast(dataHoraCotacao AS TIMESTAMP)"
        )
    )
    .withColumn(
        "cotacao_dolar",
        F.expr(
            "try_cast(cotacaoCompra "
            "AS DECIMAL(18,6))"
        )
    )
    .withColumn(
        "data_cotacao",
        F.to_date("instante")
    )
    .filter(
        F.col("data_cotacao").isNotNull()
        & (F.col("cotacao_dolar") > 0)
    )
)

# Última cotação de cada dia.
janela_dia = (
    Window.partitionBy("data_cotacao")
    .orderBy(
        F.col("instante").desc(),
        F.col("ingestion_datetime").desc()
    )
)

diaria = (
    df.withColumn(
        "_rn",
        F.row_number().over(janela_dia)
    )
    .filter(F.col("_rn") == 1)
    .select(
        "data_cotacao", "cotacao_dolar"
    )
)

# Gerar calendário contínuo desde a primeira
# cotação disponível até o dia atual.
limites = diaria.agg(
    F.min("data_cotacao").alias("inicio"),
    F.greatest(
        F.max("data_cotacao"),
        F.current_date()
    ).alias("fim")
)

calendario = limites.select(
    F.explode(
        F.sequence("inicio", "fim")
    ).alias("data_cotacao")
)

serie = calendario.join(
    diaria,
    "data_cotacao",
    "left"
)

# Forward Fill com a última cotação válida.
janela_ffill = (
    Window.orderBy("data_cotacao")
    .rowsBetween(
        Window.unboundedPreceding, 0
    )
)

resultado = serie.withColumn(
    "cotacao_dolar",
    F.last(
        "cotacao_dolar",
        ignorenulls=True
    ).over(janela_ffill)
)

salvar_silver(
    resultado, "tb_cotacao_dolar"
)

display(
    resultado.orderBy("data_cotacao")
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


data_cotacao,cotacao_dolar
2026-09-30,5.180300
2026-10-01,5.207300
2026-10-02,5.223200
2026-10-03,5.223200
2026-10-04,5.223200
2026-10-05,4.985300
2026-10-06,4.969200
2026-10-07,4.969200


# 1.3.2 — Dados Financeiros dos Filmes

**Origem:** `bronze.tb_movies_financials`  
**Destino:** `silver.tb_financeiro_filmes`

**Regras:** padronizar `id → id_filme`, `budget → orcamento_usd`, `revenue → receita_usd`; tratar ausências, símbolos, separadores e sufixos `K/M/B`. Valores menores ou iguais a zero tornam-se `NULL` e os válidos são convertidos em `DECIMAL(18,2)`.

**Cálculos:** converter orçamento e receita em BRL usando a **última PTAX disponível na carga** (não a cotação histórica do lançamento); calcular lucro (`receita - orçamento`) em USD/BRL e margem (`lucro / orçamento × 100`). Operações inválidas ou sem base suficiente permanecem `NULL`.


In [0]:
# Função para higienizar valores monetários.
def tratar_moeda(df, origem, destino):

    texto = F.lower(
        F.regexp_replace(
            F.trim(F.col(origem)),
            r"(?i)(usd|brl|us\$|r\$|\$|€|£|\s)",
            ""
        )
    )

    # Fatores de abreviação.
    fator = (
        F.when(texto.rlike("k$"), 1000.0)
        .when(texto.rlike("m$"), 1000000.0)
        .when(texto.rlike("b$"), 1000000000.0)
        .otherwise(1.0)
    )

    numero = F.regexp_replace(
        texto, r"[kmb]$", ""
    )

    # Identificar padrões de separadores.
    numero = (
        F.when(
            numero.rlike(
                r"^-?\d{1,3}(,\d{3})+(\.\d+)?$"
            ),
            F.regexp_replace(numero, ",", "")
        )
        .when(
            numero.rlike(
                r"^-?\d{1,3}(\.\d{3})+(,\d+)?$"
            ),
            F.regexp_replace(
                F.regexp_replace(
                    numero, r"\.", ""
                ),
                ",", "."
            )
        )
        .otherwise(
            F.regexp_replace(
                numero, ",", "."
            )
        )
    )

    df = df.withColumn("_numero", numero)

    df = df.withColumn(
        "_valor",
        F.expr(
            "try_cast(_numero AS DOUBLE)"
        )
    )

    df = df.withColumn(
        "_total",
        F.col("_valor") * fator
    )

    df = df.withColumn(
        destino,
        F.when(
            F.col("_total") > 0,
            F.expr(
                "try_cast(_total "
                "AS DECIMAL(18,2))"
            )
        )
    )

    return df.drop(
        "_numero", "_valor", "_total"
    )


# Ler origem e remover duplicatas por filme.
df = mais_recente(
    spark.table("bronze.tb_movies_financials")
)

df = tratar_moeda(
    df, "budget", "orcamento_usd"
)

df = tratar_moeda(
    df, "revenue", "receita_usd"
)

# Obter a cotação mais recente.
cotacao = (
    spark.table("silver.tb_cotacao_dolar")
    .orderBy(
        F.col("data_cotacao").desc()
    )
    .select("cotacao_dolar")
    .first()
)

if cotacao is None or cotacao[0] is None:
    raise ValueError("Cotação não disponível.")

taxa = F.lit(cotacao[0])

# Conversões e indicadores.
df = (
    df
    .withColumn(
        "orcamento_brl",
        (F.col("orcamento_usd") * taxa)
        .cast("decimal(18,2)")
    )
    .withColumn(
        "receita_brl",
        (F.col("receita_usd") * taxa)
        .cast("decimal(18,2)")
    )
    .withColumn(
        "lucro_usd",
        (
            F.col("receita_usd")
            - F.col("orcamento_usd")
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "lucro_brl",
        (
            F.col("receita_brl")
            - F.col("orcamento_brl")
        ).cast("decimal(18,2)")
    )
    .withColumn(
        "margem_lucro_percentual",
        F.when(
            F.col("orcamento_usd") > 0,
            F.round(
                F.col("lucro_usd")
                / F.col("orcamento_usd")
                * 100,
                2
            )
        )
    )
)

resultado = df.select(
    F.col("id").alias("id_filme"),
    "orcamento_usd",
    "receita_usd",
    "orcamento_brl",
    "receita_brl",
    "lucro_usd",
    "lucro_brl",
    "margem_lucro_percentual"
)

salvar_silver(
    resultado,
    "tb_financeiro_filmes"
)

display(resultado.limit(20))

id_filme,orcamento_usd,receita_usd,orcamento_brl,receita_brl,lucro_usd,lucro_brl,margem_lucro_percentual
1000004,null,null,null,null,null,null,null
1000005,null,null,null,null,null,null,null
1000007,null,null,null,null,null,null,null
1000011,null,null,null,null,null,null,null
1000014,null,null,null,null,null,null,null
1000030,null,null,null,null,null,null,null
1000054,null,null,null,null,null,null,null
1000058,4700000.00,null,23355240.00,null,null,null,null
1000059,null,null,null,null,null,null,null
1000073,6000000.00,null,29815200.00,null,null,null,null


# 1.3.3 — Métricas de Engajamento

**Origem:** `bronze.tb_movies_metrics`  
**Destino:** `silver.tb_metricas_engajamento`

| Origem | Destino |
|---|---|
| `id` | `id_filme` |
| `popularity` | `popularidade` |
| `vote_average` / `vote_count` | `nota_media_tmdb` / `qtd_votos_tmdb` |
| `averageRating` / `numVotes` | `nota_media_imdb` / `qtd_votos_imdb` |

**Regras:** padronizar separadores, usar `try_cast` para tipos numéricos, invalidar notas fora de `0–10` e métricas negativas. Sinais combinados de **Column Shift** (ano em `popularity` e texto em `averageRating`) invalidam a popularidade suspeita, sem impor teto arbitrário ao índice.


In [0]:
# Função para normalizar decimais.
# Em popularidade/notas, uma vírgula isolada
# será interpretada como separador decimal.
def normalizar_decimal(c):

    x = F.regexp_replace(
        F.trim(c), r"\s", ""
    )

    return (
        F.when(
            x.rlike(
                r"^-?\d{1,3}(\.\d{3})+,\d+$"
            ),
            F.regexp_replace(
                F.regexp_replace(
                    x, r"\.", ""
                ),
                ",", "."
            )
        )
        .when(
            x.rlike(
                r"^-?\d{1,3}(,\d{3})+\.\d+$"
            ),
            F.regexp_replace(x, ",", "")
        )
        .otherwise(
            F.regexp_replace(x, ",", ".")
        )
    )


# Para contagens inteiras, aceita
# separadores de milhar válidos.
def normalizar_inteiro(c):

    x = F.regexp_replace(
        F.trim(c), r"\s", ""
    )

    return (
        F.when(
            x.rlike(
                r"^-?\d{1,3}(,\d{3})+$"
            ),
            F.regexp_replace(x, ",", "")
        )
        .when(
            x.rlike(
                r"^-?\d{1,3}(\.\d{3})+$"
            ),
            F.regexp_replace(x, r"\.", "")
        )
        .otherwise(x)
    )


# Ler dados e remover duplicatas.
df = mais_recente(
    spark.table("bronze.tb_movies_metrics")
)

# Popularidade e notas.
decimais = {
    "popularity": "popularidade",
    "vote_average": "nota_media_tmdb",
    "averageRating": "nota_media_imdb"
}

for origem, destino in decimais.items():

    df = df.withColumn(
        "_valor",
        normalizar_decimal(F.col(origem))
    )

    df = df.withColumn(
        destino,
        F.expr(
            "try_cast(_valor AS DOUBLE)"
        )
    ).drop("_valor")


# Quantidades de votos.
inteiros = {
    "vote_count": "qtd_votos_tmdb",
    "numVotes": "qtd_votos_imdb"
}

for origem, destino in inteiros.items():

    df = df.withColumn(
        "_valor",
        normalizar_inteiro(F.col(origem))
    )

    df = df.withColumn(
        destino,
        F.expr(
            "try_cast(_valor AS INT)"
        )
    ).drop("_valor")


# Validar as notas na escala de 0 a 10.
for coluna in [
    "nota_media_tmdb",
    "nota_media_imdb"
]:
    df = df.withColumn(
        coluna,
        F.when(
            F.col(coluna).between(0, 10),
            F.col(coluna)
        )
    )


# Invalidar métricas negativas.
for coluna in [
    "popularidade",
    "qtd_votos_tmdb",
    "qtd_votos_imdb"
]:
    df = df.withColumn(
        coluna,
        F.when(
            F.col(coluna) >= 0,
            F.col(coluna)
        )
    )

# Um ano isolado na popularidade, combinado
# com texto na nota IMDb, indica Column Shift.
ano_suspeito = (
    F.trim(F.col("popularity"))
    .rlike(r"^(19|20)\d{2}$")
)

# Desconsiderar marcadores comuns de ausência.
texto_imdb = F.lower(
    F.trim(F.col("averageRating"))
)

texto_deslocado = (
    texto_imdb.rlike(r"[a-z]")
    & ~texto_imdb.isin(
        "unknown",
        "n/a",
        "none",
        "null",
        "não informado"
    )
)

# Invalidar apenas quando os dois indícios coexistem.
df = df.withColumn(
    "popularidade",
    F.when(
        ano_suspeito & texto_deslocado,
        F.lit(None).cast("double")
    ).otherwise(
        F.col("popularidade")
    )
)

resultado = df.select(
    F.col("id").alias("id_filme"),
    "popularidade",
    "nota_media_tmdb",
    "qtd_votos_tmdb",
    "nota_media_imdb",
    "qtd_votos_imdb"
)

salvar_silver(
    resultado,
    "tb_metricas_engajamento"
)

display(resultado.limit(20))


id_filme,popularidade,nota_media_tmdb,qtd_votos_tmdb,nota_media_imdb,qtd_votos_imdb
1000004,1.132,0.0,0,6.8,27
1000005,0.6,0.0,0,null,40
1000007,0.6,0.0,0,4.7,10
1000011,1.169,0.0,0,4.8,16
1000014,0.6,0.0,0,7.2,15
1000030,0.615,0.0,0,null,25
1000054,0.6,0.0,0,5.9,10
1000058,1.489,6.75,6,6.2,382
1000059,0.6,0.0,0,7.7,25
1000073,13.212,6.8,15,null,236


## 1.3.3.1 — Validação de Métricas

Confirmar que nenhuma métrica armazenada viola os limites de negócio e quantificar valores ausentes resultantes da conversão ou da invalidação.

In [0]:
# Verificar limites de negócio: esperado zero registros.
violacoes = spark.sql("""
    SELECT *
    FROM silver.tb_metricas_engajamento
    WHERE nota_media_tmdb NOT BETWEEN 0 AND 10
       OR nota_media_imdb NOT BETWEEN 0 AND 10
       OR popularidade < 0
       OR qtd_votos_tmdb < 0
       OR qtd_votos_imdb < 0
""")
assert violacoes.count() == 0, "Métricas fora dos limites válidos"

# Quantificar a ausência de métricas após a tipagem segura.
display(spark.sql("""
    SELECT
        COUNT(*) AS total,
        COUNT(*) FILTER (WHERE popularidade IS NULL) AS sem_popularidade,
        COUNT(*) FILTER (WHERE nota_media_tmdb IS NULL) AS sem_nota_tmdb,
        COUNT(*) FILTER (WHERE nota_media_imdb IS NULL) AS sem_nota_imdb
    FROM silver.tb_metricas_engajamento
"""))


total,sem_popularidade,sem_nota_tmdb,sem_nota_imdb
99013,4174,3528,13201


# 1.3.4 — Avaliações dos Usuários

**Origem:** `bronze.tb_movies_reviews`  
**Destino:** `silver.tb_avaliacoes_usuarios`

Mapear `id`, `nome`, `nota` e `comentario` para `id_filme`, `nome_usuario`, `nota_usuario` e `comentario_usuario`. Converter notas com `try_cast`, manter apenas `0–10`, preencher comentários vazios com `Sem comentário` e remover avaliações duplicadas pela combinação dos quatro campos.


In [0]:
df = spark.table(
    "bronze.tb_movies_reviews"
)

# Deduplicação baseada nos campos brutos.
df = df.dropDuplicates([
    "id", "nome", "nota", "comentario"
])

# Renomear e converter os campos.
resultado = df.select(
    F.col("id").alias("id_filme"),
    F.col("nome").alias("nome_usuario"),
    F.expr(
        "try_cast("
        "replace(trim(nota), ',', '.') "
        "AS DOUBLE)"
    ).alias("nota_usuario"),
    F.col("comentario").alias(
        "comentario_usuario"
    )
)

# Validar notas no intervalo de 0 a 10.
resultado = resultado.withColumn(
    "nota_usuario",
    F.when(
        F.col("nota_usuario").between(0, 10),
        F.col("nota_usuario")
    )
)

# Tratar comentários vazios.
resultado = resultado.withColumn(
    "comentario_usuario",
    F.when(
        F.col("comentario_usuario").isNull()
        | (
            F.trim("comentario_usuario") == ""
        ),
        F.lit("Sem comentário")
    ).otherwise(
        F.trim("comentario_usuario")
    )
)

# Evitar duplicatas criadas pela padronização
# de notas inválidas e comentários ausentes.
resultado = resultado.dropDuplicates([
    "id_filme", "nome_usuario",
    "nota_usuario", "comentario_usuario"
])

salvar_silver(
    resultado,
    "tb_avaliacoes_usuarios"
)

display(resultado.limit(20))


id_filme,nome_usuario,nota_usuario,comentario_usuario
442113,Mariana Cardoso 277,4.4,Sem comentário
637007,Lucas Reis 602,3.9,Sem comentário
449479,Sérgio Freitas,0.7,Péssimo em todos os sentidos.
413036,Gabriela Monteiro 401,7.5,Sem comentário
528480,Leonardo Monteiro,6.3,Assisti até o final mas não me marcou.
387727,Maria Alves 707,8.2,"Gostei bastante, recomendo."
1032506,Amanda Castro 242,4.7,"Fraco, não recomendo."
446554,Sandra Rodrigues 736,6.5,Assisti até o final mas não me marcou.
569916,Camila Lima 489,8.2,Muito bom! Vale a pena assistir.
864552,Roberto Almeida 555,9.3,"Obra-prima do cinema, simplesmente espetacular."


# 1.3.5 — Normalização dos Gêneros

**Origem:** `bronze.tb_credits_and_tags` (`genres`)  
**Destino:** `silver.tb_generos`

Dividir os gêneros com `split` e `explode`, reconhecendo os separadores `,`, `;` e `|`; normalizar grafia, validar o domínio TMDB e remover pares repetidos de filme/gênero. Processar todas as linhas de créditos antes de deduplicar, pois registros adicionais podem conter gêneros complementares.

**Grão:** uma associação única entre filme e gênero, usada na ponte Gold.


In [0]:
# Considerar todos os registros de créditos,
# pois linhas repetidas de um filme podem
# conter gêneros complementares.
df = spark.table(
    "bronze.tb_credits_and_tags"
)

# Domínio conhecido de gêneros do TMDB.
generos_validos = [
    "Action", "Adventure", "Animation",
    "Comedy", "Crime", "Documentary",
    "Drama", "Family", "Fantasy",
    "History", "Horror", "Music",
    "Mystery", "Romance", "Science Fiction",
    "TV Movie", "Thriller", "War",
    "Western"
]

# Dicionário para padronizar maiúsculas.
mapa = F.create_map(*[
    item
    for genero in generos_validos
    for item in (
        F.lit(genero.lower()),
        F.lit(genero)
    )
])

resultado = (
    df
    .select(
        F.col("id").alias("id_filme"),

        F.explode(
            F.split(
                F.col("genres"),
                r"[,;|]"
            )
        ).alias("genero_bruto")
    )
    .withColumn(
        "_chave",
        F.lower(
            F.trim("genero_bruto")
        )
    )
    .withColumn(
        "nome_genero",
        F.element_at(
            mapa,
            F.col("_chave")
        )
    )
    .filter(
        F.col("nome_genero").isNotNull()
    )
    .select(
        "id_filme",
        "nome_genero"
    )
    .dropDuplicates()
)

salvar_silver(resultado, "tb_generos")

display(resultado.limit(20))


id_filme,nome_genero
329865,Mystery
269149,Animation
429617,Science Fiction
330459,Action
127380,Family
332562,Romance
291805,Action
293167,Action
381284,History
454626,Action


# 1.3.6 — Pessoas e Empresas

**Origem:** `bronze.tb_credits_and_tags`  
**Destino:** `silver.tb_pessoas_empresas`

| Coluna | `tipo_entidade` |
|---|---|
| `cast` | `Ator` |
| `directors` | `Diretor` |
| `writers` | `Roteirista` |
| `production_companies` | `Produtora` |

Aplicar `split`/`explode`, consolidar participantes e empresas, normalizar capitalização e remover resíduos comuns de **Column Shift** e duplicatas pelo trio `(id_filme, nome_entidade, tipo_entidade)`. Conservar linhas adicionais do mesmo filme antes de extrair associações válidas.

**Limitação:** filtros heurísticos podem descartar nomes longos legítimos e não corrigem todos os campos deslocados.


In [0]:
# Preservar participações complementares
# que podem existir em linhas adicionais.
df = spark.table(
    "bronze.tb_credits_and_tags"
)

# Extração de entidades por coluna.
def extrair_entidades(df, coluna, tipo):

    return (
        df.select(
            F.col("id").alias("id_filme"),

            F.explode(
                F.split(
                    F.regexp_replace(
                        F.col(coluna),
                        ";", ","
                    ),
                    ","
                )
            ).alias("nome_entidade")
        )
        .withColumn(
            "nome_entidade",
            F.initcap(
                F.lower(
                    F.trim(
                        F.regexp_replace(
                            "nome_entidade",
                            r"\s+", " "
                        )
                    )
                )
            )
        )
        .withColumn(
            "tipo_entidade",
            F.lit(tipo)
        )
    )


# Extração individual.
atores = extrair_entidades(
    df, "cast", "Ator"
)

diretores = extrair_entidades(
    df, "directors", "Diretor"
)

roteiristas = extrair_entidades(
    df, "writers", "Roteirista"
)

produtoras = extrair_entidades(
    df, "production_companies", "Produtora"
)


# Consolidação das entidades.
resultado = (
    atores
    .unionByName(diretores)
    .unionByName(roteiristas)
    .unionByName(produtoras)
)

# Remover resíduos comuns de Column Shift.
resultado = (
    resultado
    .filter(
        F.col("nome_entidade").isNotNull()
    )
    .filter(
        F.length("nome_entidade") >= 2
    )
    .filter(
        F.length("nome_entidade") <= 80
    )
    .filter(
        F.size(
            F.split("nome_entidade", " ")
        ) <= 8
    )
    .filter(
        ~F.col("nome_entidade").rlike(
            r"^\d+(\.\d+)?$"
        )
    )
    .filter(
        ~F.lower("nome_entidade").isin(
            "n/a", "null",
            "none", "unknown",
            "não informado"
        )
    )
    .dropDuplicates([
        "id_filme",
        "nome_entidade",
        "tipo_entidade"
    ])
)

salvar_silver(
    resultado,
    "tb_pessoas_empresas"
)

display(resultado.limit(20))

id_filme,nome_entidade,tipo_entidade
299534,Josh Brolin,Ator
475557,Zazie Beetz,Ator
354912,Alfonso Arau,Ator
381288,Izzie Coffey,Ator
269149,J.k. Simmons,Ator
299537,Brie Larson,Ator
181808,Domhnall Gleeson,Ator
141052,Gal Gadot,Ator
466272,Timothy Olyphant,Ator
277834,Troy Polamalu,Ator


# Validação da Camada Silver

Confirmar a existência das sete tabelas, tipos de dados e unicidade de filme em informações, financeiro e métricas. Para cotação, validar ausência de `NULL` e cobertura de todos os dias entre a primeira e a última data da série.


In [0]:
tabelas = [
    "tb_info_filmes",
    "tb_financeiro_filmes",
    "tb_metricas_engajamento",
    "tb_avaliacoes_usuarios",
    "tb_generos",
    "tb_pessoas_empresas",
    "tb_cotacao_dolar"
]

for tabela in tabelas:

    df = spark.table(f"silver.{tabela}")

    print(
        f"{tabela}: {df.count()} registros"
    )

    df.printSchema()


# Verificação de unicidade por filme.
tabelas_unicas = [
    "tb_info_filmes",
    "tb_financeiro_filmes",
    "tb_metricas_engajamento"
]

for tabela in tabelas_unicas:

    df = spark.table(f"silver.{tabela}")

    duplicados = (
        df.groupBy("id_filme")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print(
        f"{tabela}: {duplicados} IDs duplicados"
    )

    assert duplicados == 0


# Verificação do Forward Fill.
cotacoes = spark.table(
    "silver.tb_cotacao_dolar"
)

assert cotacoes.filter(
    F.col("cotacao_dolar").isNull()
).count() == 0

# Validar continuidade e ausência de duplicatas na série diária.
resumo_cotacao = cotacoes.agg(
    F.min("data_cotacao").alias("inicio"),
    F.max("data_cotacao").alias("fim"),
    F.countDistinct("data_cotacao").alias("qtd_dias")
).first()

assert resumo_cotacao["inicio"] is not None

dias_esperados = (
    resumo_cotacao["fim"] - resumo_cotacao["inicio"]
).days + 1

assert resumo_cotacao["qtd_dias"] == dias_esperados
assert cotacoes.count() == dias_esperados

print("Validações Silver concluídas!")


tb_info_filmes: 97879 registros
root
 |-- id_filme: string (nullable = true)
 |-- titulo: string (nullable = true)
 |-- titulo_original: string (nullable = true)
 |-- data_lancamento: date (nullable = true)
 |-- duracao_minutos: integer (nullable = true)
 |-- idioma_original: string (nullable = true)
 |-- status_filme: string (nullable = true)
 |-- sinopse: string (nullable = true)
 |-- frase_divulgacao: string (nullable = true)
 |-- ano_lancamento: integer (nullable = true)

tb_financeiro_filmes: 99006 registros
root
 |-- id_filme: string (nullable = true)
 |-- orcamento_usd: decimal(18,2) (nullable = true)
 |-- receita_usd: decimal(18,2) (nullable = true)
 |-- orcamento_brl: decimal(18,2) (nullable = true)
 |-- receita_brl: decimal(18,2) (nullable = true)
 |-- lucro_usd: decimal(18,2) (nullable = true)
 |-- lucro_brl: decimal(18,2) (nullable = true)
 |-- margem_lucro_percentual: decimal(25,2) (nullable = true)

tb_metricas_engajamento: 99013 registros
root
 |-- id_filme: string (null